# 06 Analysis
Heat exposure and training timing, compared between **patients**, **healthy runners** and **pacers**.

In [ ]:
import sys
sys.path.append("../src")

from heatwatch import dataset
from heatwatch.analysis import compare, heat_exposure

## Settings
The settings of this analysis

In [ ]:
s = compare.Settings(
    params=["resting_hr", "hr", "wbgt"],   # parameters a participant needs (names as in dataset.PARAMETERS); hr / wbgt /
                                           # location: in the same training, in the window_days before the race;
                                           # the others: any value in the participant's data
    within_hours=1,       # training timing: share of trainings starting within this many hours of the race start
    min_group_n=5,        # groups with fewer values are described but not tested
    window_days=14,       # acclimatization window: only trainings in this many days before the race used
    decay_per_day=0.023,  # HEI: ~2.3 % loss of heat adaptation per day
    race="MR",            # race used when a participant ran both (otherwise the race they ran)
)
print(f"parameters: {', '.join(s.params)}; within {s.within_hours:g} h; groups tested from n = {s.min_group_n}; "
      f"{s.window_days} days before the race; decay {s.decay_per_day:g} per day; {s.race} when both were run")

## Load
The dataset of step 04.

In [ ]:
ds = dataset.load()
ds

## Heat dose per training
Trainings in the 14 days before the race with mean HR, WBGT and resting HR.

In [ ]:
hd = heat_exposure.heat_doses(ds.trainings, ds.participants, compare.races_used(ds, s), s.window_days,
                              s.decay_per_day)   # both races together
hd[["study_id", "date", "location", "duration_min", "hr_mean", "hr_rest", "hr_max_used", "dHR", "effort",
    "wbgt_mean", "wbgt_excess", "heat_dose", "days_before_race", "hei_part"]].head(10)

In [ ]:
print(f"{len(hd)} trainings with a heat dose, of which {(hd.wbgt_excess > 0).sum()} above "
      f"{heat_exposure.THRESHOLD:g} C WBGT ({(hd.location == 'indoor').sum()} indoor)")

## Resting HR fallback

In [ ]:
res = compare.run(ds, s, save=False)
fb = res.fallback
print(f"{(fb.with_fallback > 0).sum()} of {len(fb)} participants with >= 1 training with the fallback, "
      f"{fb.only_fallback.sum()} only with the fallback; {fb.with_fallback.sum()} of {fb.trainings_with_dose.sum()} trainings")
fb[fb.with_fallback > 0]

In [ ]:
fb[fb.only_fallback]

In [ ]:
res.fallback_trainings

## Participants in the analysis
Exclusions of step 05, then (first that applies): no training in the 14 days before the race, no resting HR at all, no training with HR and WBGT in the 14 days before the race.

In [ ]:
res.flow

## Total comparison

In [ ]:
res.participants

In [ ]:
res.table

**Patients vs healthy runners**:

In [ ]:
res.patients_vs_healthy

## Boxplots
Three figures: timing direction (training start - race start, < 0 = earlier; % more than 1 h earlier / later); heat exposure (% exposed, HEI of the exposed); the three factors of HEI (HD = duration × effort × WBGT above 14 °C) separately: total training duration, effort and WBGT of the trainings in the 14 days before the race.

In [ ]:
res.figures["direction"]    # timing direction: earlier or later than the race start time

In [ ]:
res.figures["heat"]         # heat exposure: % exposed and HEI of the exposed

In [ ]:
res.figures["components"]   # the three factors of HEI: duration, effort, WBGT

## Per race

### Marathon Rotterdam

In [ ]:
res_mr = compare.run(ds, s, "MR", save=False)
res_mr.flow

In [ ]:
res_mr.table

**Patients vs healthy runners**: planned comparison of only these two groups, every outcome (Mann-Whitney U, r = rank-biserial: > 0 higher in patients; exposed: Fisher's exact, odds ratio).

In [ ]:
res_mr.patients_vs_healthy

In [ ]:
res_mr.figures["direction"]    # timing direction: earlier or later than the race start time

In [ ]:
res_mr.figures["heat"]         # heat exposure: % exposed and HEI of the exposed

In [ ]:
res_mr.figures["components"]   # the three factors of HEI: duration, effort, WBGT

### CPC

In [ ]:
res_cpc = compare.run(ds, s, "CPC", save=False)
res_cpc.flow

In [ ]:
res_cpc.table

**Patients vs healthy runners**: planned comparison of only these two groups, every outcome (Mann-Whitney U, r = rank-biserial: > 0 higher in patients; exposed: Fisher's exact, odds ratio).

In [ ]:
res_cpc.patients_vs_healthy

In [ ]:
res_cpc.figures["direction"]    # timing direction: earlier or later than the race start time

In [ ]:
res_cpc.figures["heat"]         # heat exposure: % exposed and HEI of the exposed

In [ ]:
res_cpc.figures["components"]   # the three factors of HEI: duration, effort, WBGT

## Save
`analysis/`: HEATWATCH_analysis.xlsx and the 3 boxplots (both races), the same ending in _MR and _CPC for the analysis per race: 3 .xlsx and 9 boxplots. Older boxplots / analysis files (old figure names, the former sensitivity analysis `_hrmax_training`) are removed first.

In [ ]:
compare.clear_old()                       # only the files made below stay in analysis/
results = {race: compare.run(ds, s, race) for race in (None, "MR", "CPC")}

## Tables for the report
Built from the results above (`compare.run`), so they match the analysis. Only study numbers.
Saved as the sheets Flow, Excluded and Characteristics in `HEATWATCH_analysis.xlsx` (the detailed list of excluded participants is in 'Excluded (detail)').

In [ ]:
from heatwatch.reports import export, paper_tables
from heatwatch.settings import ANALYSIS

res, res_mr, res_cpc = results[None], results["MR"], results["CPC"]
flow = paper_tables.flow(res)                 # a) flow of inclusion
flow

In [ ]:
excluded = paper_tables.excluded(res)         # b) excluded participants, first reason that applies
excluded

Ran both races, excluded from one per-race analysis but included in the main analysis:

In [ ]:
both = paper_tables.ran_both(res, res_mr, res_cpc)
both

In [ ]:
characteristics = paper_tables.characteristics(res, ds)   # c) included participants
characteristics

In [ ]:
export.save_report_tables({
    "Flow": [("Flow of inclusion", "both races, every participant once; negative = excluded at that step", flow)],
    "Excluded": [("Excluded participants", "first reason that applies, in the order 1-8", excluded),
                 ("Ran both races", "excluded from one per-race analysis, included in the main analysis", both)],
    "Characteristics": [("Characteristics of the included participants", "n (%) or median (IQR); percentages rounded half up", characteristics)]})

### Checks

In [ ]:
paper_tables.checks(res, flow, excluded, characteristics, ANALYSIS / "HEATWATCH_analysis.xlsx")